# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides an example workflow for loading and exploring the [FAIR²](https://doi.org/10.71728/senscience.vq4a-28xa) mass spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")

## 2. Data Overview
Review available record sets, their IDs (using `@id`), and their fields and columns. 

In Croissant, each data table is a `RecordSet` identified by its `@id`.

In [ ]:
print("\nAvailable Record Sets:")
record_sets = metadata.record_sets
for rs in record_sets:
    print(f"  - Name: {rs.name}")
    print(f"    @id: {rs.id}")
    field_ids = [f.id for f in rs.fields]
    fields_str = ', '.join(field_ids)
    print(f"    Fields (@id): {fields_str}\n")

# Let's print the fields, columns, and their @id for the first record set as an example
if record_sets:
    first_rs = record_sets[0]
    print(f"RecordSet '{first_rs.name}' columns:")
    for field in first_rs.fields:
        cols = getattr(field, 'columns', [])
        col_ids = [c.id for c in cols]
        print(f"  - Field: {field.name} @id: {field.id}, Columns: {col_ids}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. 
Use the **`@id`** values from the previous overview to extract data.

In [ ]:
# We'll extract ALL record sets for completeness
dataframes = {}
for rs in record_sets:
    rs_id = rs.id
    print(f"Loading records for RecordSet: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"  Loaded {len(df)} rows with columns: {df.columns.tolist()}")
    else:
        print("  No records found.")

# For demonstration, pick first non-empty dataframe
primary_rs_id = None
for rs_id, df in dataframes.items():
    if not df.empty:
        primary_rs_id = rs_id
        break

if primary_rs_id:
    print(f"\nUsing RecordSet: {primary_rs_id} for analysis.")
    print(dataframes[primary_rs_id].head())
else:
    print("No non-empty record sets found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps:
- Filtering records based on a numeric field
- Normalizing data
- Grouping by a categorical ID

We use fields' `@id` for all references.

Pick a numeric and a group field. Adjust as needed for your use case.

In [ ]:
if primary_rs_id:
    df = dataframes[primary_rs_id]

    # Show column names so user can reference fields by `@id`
    print(f"Columns in RecordSet {primary_rs_id}: {df.columns.tolist()}")

    # Try to find a common numeric field
    # If not sure, print available columns
    # Examples: 'cr:abundance_sum', 'cr:peptides_count', 'cr:mw_kda', etc.
    candidate_numeric_fields = [col for col in df.columns if df[col].dtype.kind in {"i", "f"}]
    if candidate_numeric_fields:
        numeric_field_id = candidate_numeric_fields[0]
    else:
        # fall back to first column
        numeric_field_id = df.columns[0]
    print(f"\nUsing numeric field (@id): {numeric_field_id}")

    # Try to find a group field
    candidate_group_fields = [col for col in df.columns if df[col].dtype == object and col != numeric_field_id]
    group_field_id = candidate_group_fields[0] if candidate_group_fields else numeric_field_id
    print(f"Using group field (@id): {group_field_id}")

    # Filtering records with threshold
    threshold = df[numeric_field_id].dropna().quantile(0.20)  # example: filter for top 80% (by value)
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtering {numeric_field_id} by threshold > {threshold:.2f}, records left: {len(filtered_df)}")

    # Normalizing
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouped means
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        grouped_df.columns = [group_field_id, f"mean_{numeric_field_id}"]
        print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the numeric field, and plot group-level aggregates.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if primary_rs_id:
    df = dataframes[primary_rs_id]
    if numeric_field_id in df.columns:
        plt.figure(figsize=(8, 4))
        sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
        plt.title(f'Distribution of {numeric_field_id}')
        plt.xlabel(numeric_field_id)
        plt.show()

    # Grouped bar plot
    if 'grouped_df' in locals() and grouped_df.shape[1] == 2:
        plt.figure(figsize=(10, 5))
        sns.barplot(x=group_field_id, y=f"mean_{numeric_field_id}", data=grouped_df)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=90)
        plt.show()

## 6. Conclusion
This notebook demonstrates how to access, explore, and process the [FAIR² Mass Spectrometry Dataset](https://doi.org/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library.

You can extend this notebook further by referencing your domain knowledge and the dataset's Croissant schema (using `@id`) to select additional fields and perform more advanced analyses.